# Exploratory Data Analysis — Diabetes Readmission

This notebook explores the raw dataset to understand its structure, missingness, and key distributions.  
We will:
1. Load the raw data
2. Standardize missing markers and schema
3. Perform exploratory data analysis (EDA)
4. Visualize Kaplan–Meier survival curves

## Step 1: Load Data
We begin by loading the raw diabetes readmission dataset.  
This step confirms the schema, variable types, and basic distributions.  
It ensures the dataset is ready for cleaning and preprocessing.

In [ ]:
# =============================================
# STEP 1 — LOAD DATA
# =============================================
df = pd.read_csv("diabetic_data.csv")
df.info()
df.describe()
df["readmitted"].value_counts()

## Step 2: Standardize Missing Markers & Schema
The dataset contains placeholders for missing values (e.g., `"?"`, `"Unknown/Invalid"`).  
We replace these with proper `NaN` values for consistency.  
We also cast key columns to numeric types and diagnosis codes to strings,  
ensuring the dataset is structured correctly for downstream analysis.

In [ ]:
# =============================================
# STEP 2 — STANDARDIZE MISSING MARKERS & SCHEMA
# =============================================
df = df.replace({"?": np.nan, "Unknown/Invalid": np.nan})

# Cast important columns as numeric
int_cols = [
    "encounter_id", "patient_nbr", "admission_type_id",
    "discharge_disposition_id", "admission_source_id",
    "time_in_hospital", "num_lab_procedures", "num_procedures",
    "num_medications", "number_outpatient", "number_emergency",
    "number_inpatient"
]

for c in int_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")

# Diagnosis codes as strings
df["diag_1"] = df["diag_1"].astype("string")
df["diag_2"] = df["diag_2"].astype("string")
df["diag_3"] = df["diag_3"].astype("string")

df.info()

## Step 5: Exploratory Data Analysis (EDA)
We explore the dataset to understand variable distributions and relationships:

- **Missingness analysis**: visualize missing data patterns with heatmaps.  
- **Distributions**: plot histograms and boxplots for key continuous variables  
  (e.g., `time_in_hospital`, `num_medications`, `number_inpatient`).  
- **Correlation heatmap**: examine correlations among numeric features.  
- **Event rates**: calculate readmission event rates by demographics (age, gender, race).

These insights help identify important predictors and guide model design.

In [ ]:
# =============================================
# STEP 5 — EDA
# =============================================
# Create an "EDA" folder if it doesn't exist
os.makedirs("EDA", exist_ok=True)

missing_percent = df.isna().mean().sort_values(ascending=False) * 100
missing_percent

# Drop high-missing columns
df = df.drop(columns=['weight', 'max_glu_serum', 'A1Cresult'])

# Event stats
df["event"].value_counts(normalize=True) * 100
df["age"].value_counts().sort_index()
df["gender"].value_counts()
df["race"].value_counts(dropna=False)

(df.groupby("gender")["event"].mean() * 100).sort_values()
(df.groupby("age")["event"].mean() * 100).sort_values()


# Correlation heatmap — NUMERIC ONLY (safe)
numeric_df = df.select_dtypes(include=["int64", "float64", "Int64"])

plt.figure(figsize=(12,10))
sns.heatmap(numeric_df.corr(), cmap="coolwarm")
plt.title("Correlation Heatmap (Numeric Features Only)")
plt.savefig("EDA/correlation_heatmap_numeric.png")
plt.show()

# Missingness heatmap
msno.heatmap(df)
plt.savefig("EDA/missingness_heatmap.png")
plt.show()

# Distribution plots
for v in ["time_in_hospital", "num_medications", "number_inpatient"]:
    plt.figure(figsize=(6, 4))
    sns.histplot(df[v], kde=True)
    plt.title(f"Distribution of {v}")
    plt.savefig(f"EDA/distribution_{v}.png")
    plt.show()

# Boxplot
plt.figure(figsize=(6, 4))
sns.boxplot(data=df, x="gender", y="num_medications")
plt.title("Num Medications by Gender")
plt.savefig("EDA/boxplot_num_medications_by_gender.png") 
plt.show()

(df.groupby("race")["event"].mean() * 100).sort_values()

## Step 6: Kaplan–Meier Curves
We estimate survival probabilities using Kaplan–Meier analysis:

- **Overall KM curve**: shows the probability of avoiding readmission over time.  
- **KM by gender**: compares survival between male and female patients.  

Interpretation:
- Steeper drops in the curve indicate higher readmission risk.  
- Differences between groups highlight potential demographic effects.  

Artifacts:
- Save KM plots to the `plots/KM_Plots/` directory for inclusion in the client report.

In [ ]:
# =============================================
# STEP 6 — KAPLAN–MEIER CURVES
# =============================================

# Create folder for KM plots
os.makedirs("KM_Plots", exist_ok=True)
km = KaplanMeierFitter()

plt.figure(figsize=(8, 5))
km.fit(df["time"], df["event"])
km.plot()
plt.title("Kaplan–Meier Curve (30-Day Readmission)")
plt.xlabel("Days")
plt.ylabel("Survival Probability")
plt.savefig("KM_Plots/kaplan_meier_curve.png")
plt.show()

# KM by Gender
plt.figure(figsize=(8, 5))
for g in ["Male", "Female"]:
    km = KaplanMeierFitter()
    mask = (df["gender"] == g)
    km.fit(df.loc[mask, "time"], df.loc[mask, "event"], label=g)
    km.plot()

plt.title("KM Curve by Gender")
plt.xlabel("Days")
plt.ylabel("Survival Probability")
plt.savefig("KM_Plots/kaplan_curve_by_gender.png")
plt.show()

# Interpretation numbers
km = KaplanMeierFitter()
km.fit(df["time"], df["event"])
print(km.survival_function_.head())
print("Survival at 30 days:", km.predict(30))

# Log-rank
male = df[df["gender"] == "Male"]
female = df[df["gender"] == "Female"]
logrank_test(male["time"], female["time"], male["event"], female["event"])